# DVCM Regression — Canonical Junction Traces

This is the **formal DVCM regression notebook** (same role as the R-THYM section in [`quickstart_notebook.ipynb`](quickstart_notebook.ipynb)). It replays the three checked-in anchors in `tests/dvcm_*_reference.json` and applies the **same tolerances** as **`tests/test_dvcm_canonical_scenarios.py`**.

| Notebook | Purpose | Source of truth |
|----------|---------|------------------|
| **This notebook** | JSON trace regression (peak, collapse time, RMS) | `dvcm_*_reference.json` |
| [`dvcm_physical_verification.ipynb`](dvcm_physical_verification.ipynb) | Independent physics formulas (mass step, collapse ΔH) | Wylie / collision estimate |
| [`dvcm_showcase.ipynb`](dvcm_showcase.ipynb) | Pedagogy: Legacy vs DVCM on a valve network | Exploratory (heavy `dt`, slow on Binder) |

**CI tolerances:** peak ≤ 0.05 ft, first collapse time ≤ 1e−9 s, RMS head ≤ 1e−9 ft.

[![Launch Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/jlillywh/RTHYM-MOC/main?labpath=examples%2Fdvcm_canonical_verification.ipynb)

## 1. Setup and reference assets

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import rthym_moc
from _verification_notebook_setup import bootstrap_verification_notebook

REPO_ROOT, TESTS_DIR = bootstrap_verification_notebook()
print(f"Repository root: {REPO_ROOT}")
print(f"rthym_moc: {getattr(rthym_moc, '__version__', 'unknown')}")
from dvcm_canonical_verification_utils import (
    CANONICAL_DT_S,
    CANONICAL_P_VAPOR_PSI,
    CASES,
    CASE_LABELS,
    COLLAPSE_TIME_ERROR_MAX_S,
    PEAK_ERROR_MAX_FT,
    RMS_TRACE_ERROR_MAX_FT,
    reference_path,
    run_and_evaluate,
)

for case_id, fname in CASES.items():
    print(f"  {CASE_LABELS[case_id]:<42} {reference_path(case_id)}")
print(f"Solver: dt={CANONICAL_DT_S} s, p_vapor={CANONICAL_P_VAPOR_PSI} psi, model=DVCM")

## 2. Run canonical scenarios

Symmetric reservoir–junction–reservoir geometry; boundary heads follow each JSON `schedule`.

In [ ]:
case_results = {}
for case_id in CASES:
    case_results[case_id] = run_and_evaluate(case_id)
print("Simulations complete.")

## 3. Validate against checked-in JSON traces

Overlays match the quickstart pattern: reference trace, simulation, and pointwise error.

In [ ]:
fig, axes = plt.subplots(len(CASES), 1, figsize=(11, 3.4 * len(CASES)), sharex=False)
if len(CASES) == 1:
    axes = [axes]
for ax, case_id in zip(axes, CASES):
    res, ref, m = case_results[case_id]
    t_ref = np.asarray(ref["time_s"], dtype=float)
    h_ref = np.asarray(ref["head_ft"], dtype=float)
    t_sim = np.asarray(res["time"], dtype=float)
    h_sim = np.asarray(res["node_head"]["J1"], dtype=float)
    ax.plot(t_ref, h_ref, "o-", color="black", markersize=5, linewidth=1.2, label="JSON reference (J1)")
    ax.plot(t_sim, h_sim, "--", color="tab:red", linewidth=1.8, label="rthym_moc simulation")
    ax.axvline(m.collapse_time_s, color="darkorange", linestyle=":", alpha=0.8, label="First collapse (sim)")
    ax.axhline(float(ref["peak_head_ft"]), color="seagreen", linestyle="-.", alpha=0.7, label="Ref peak")
    ok = "PASS" if m.passed else "FAIL"
    ax.set_title(f"{CASE_LABELS[case_id]} — RMS {m.rms_head_error_ft:.2e} ft [{ok}]")
    ax.set_ylabel("J1 head (ft)")
    ax.legend(loc="best", fontsize=8)
    ax.grid(True, alpha=0.3)
axes[-1].set_xlabel("Time (s)")
fig.suptitle("DVCM canonical regression: simulated vs checked-in reference heads", fontweight="bold", y=1.01)
fig.tight_layout()
plt.show()

### 3a. Pointwise head error (sim − reference)

In [ ]:
for case_id in CASES:
    res, ref, m = case_results[case_id]
    t_ref = np.asarray(ref["time_s"], dtype=float)
    h_sim = np.asarray(res["node_head"]["J1"], dtype=float)
    err = h_sim[: t_ref.size] - np.asarray(ref["head_ft"], dtype=float)
    fig, ax = plt.subplots(figsize=(10, 2.8))
    ax.plot(t_ref, err, color="firebrick", linewidth=1.2)
    ax.axhline(0.0, color="black", linewidth=0.8)
    ax.set_title(f"{CASE_LABELS[case_id]} — max |error| {np.max(np.abs(err)):.2e} ft")
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Error (ft)")
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    plt.show()

## 4. Cavity volume and collapse flags (reference fields in JSON)

In [ ]:
case_id = "rapid_closure"
res, ref, m = case_results[case_id]
t = np.asarray(res["time"], dtype=float)
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
axes[0].plot(ref["time_s"], ref["cavity_volume_ft3"], "o--", color="gray", label="Reference volume")
axes[0].plot(t, res["node_cavity_volume"]["J1"], color="darkcyan", label="Simulated volume")
axes[0].set_ylabel("Cavity volume (ft³)")
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].step(ref["time_s"], ref["collapse_flag"], where="mid", color="gray", label="Reference collapse")
axes[1].step(t, res["node_cavity_collapse_flag"]["J1"], where="mid", color="crimson", label="Sim collapse")
axes[1].set_xlabel("Time (s)")
axes[1].set_ylabel("Collapse flag")
axes[1].legend()
axes[1].grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

In [ ]:
res, ref, m = case_results["long_run"]
t = np.asarray(res["time"], dtype=float)
fig, ax = plt.subplots(figsize=(10, 2.8))
ax.step(ref["time_s"], ref["collapse_flag"], where="mid", color="gray", label="Reference")
ax.step(t, res["node_cavity_collapse_flag"]["J1"], where="mid", color="crimson", label="Simulation")
ax.set_title(f"Long run — {m.collapse_events} sim collapses vs {m.reference_collapse_events} reference")
ax.set_xlabel("Time (s)")
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
plt.show()

## 5. Pass/fail summary (pytest tolerances)

In [ ]:
print(f"{'Case':<44} {'Peak err':>10} {'Δt_coll':>12} {'RMS err':>12} {'Extra':>6} {'PASS':>6}")
print("-" * 92)
for case_id in CASES:
    m = case_results[case_id][2]
    print(
        f"{CASE_LABELS[case_id]:<44} {m.peak_head_error_ft:10.3e} {m.collapse_time_error_s:12.3e} "
        f"{m.rms_head_error_ft:12.3e} {str(m.passed_extra):>6} {str(m.passed):>6}"
    )
print(f"\nTolerances: peak ≤ {PEAK_ERROR_MAX_FT} ft, collapse time ≤ {COLLAPSE_TIME_ERROR_MAX_S:g} s, RMS ≤ {RMS_TRACE_ERROR_MAX_FT:g} ft")
all_pass = all(case_results[c][2].passed for c in CASES)
print("Overall:", "PASS" if all_pass else "FAIL")